# Build Forcing Validation - Humidity
- Tony Grove

In [1]:
%matplotlib ipympl
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from buildforcing.datasets import PNNLSnotel
import os

In [2]:
# Run parameters
site = 'Tony Grove RS'
latitude = 41.89
longitude = -111.57
start_date = '20151001'
end_date = '20161001'
buildforcing_version = 'v0.4.0'
environment = '/home/chriscox/Data/'
#environment = 'C:/Users/clmbn/NMT_PhD/data/' # Windows

In [3]:
# Create a data extraction function for forcing data
def get_forcing_values(forcing_file, variable, start_date, end_date) -> pd.DataFrame:
    """
    Extract forcing values from a NetCDF file for a given variable and date range.
    
    Parameters:
        forcing_file (str): Path to the NetCDF file.
        variable (str): Variable name to extract.
        start_date (str): Start date in 'YYYYMMDD' format.
        end_date (str): End date in 'YYYYMMDD' format.

    Returns:
        pd.DataFrame: DataFrame containing the extracted values.
    """
    ds = xr.open_dataset(forcing_file)
    subset = ds[variable].isel(latitude=0, longitude=0)
    df: pd.DataFrame = subset.to_dataframe()
    ds.close() # Seems to help when running in Jupyter Notebook

    df = df.loc[start_date:end_date]
    
    return df

### Load datasets

In [4]:
# Load the forcing and NLDAS datasets
sitename = site.replace(' ', '').lower()
forcing = xr.open_dataset(os.path.join(environment, 'forcing', f'{sitename}_{start_date}_{end_date}_{buildforcing_version}.nc'))

In [5]:
nldas = xr.open_dataset(os.path.join(environment, 'nldas', f'NLDAS_{latitude}_{longitude}_{start_date}_{end_date}.nc'))

In [6]:
snotel = PNNLSnotel(site_name=site, storage_path=os.path.join(environment, 'snotel'))

### GAMUT Data
In order to compare specific humidity, I need to use temperature to convert relative humidity to specific humidity. I am using the Rotronic HC2S3 GAMUT data from RH. I know it also measures temperature, but I didn't download that data, plus it might be questionable since I remember that sensor stuck out the side of the radiation shield. I will use the ST110 thermister data for temperature.

In [ ]:
gamut_st110 = pd.read_csv(
    os.path.join(environment, 'gamut/tonygrove', f'LR_TG_C_AirTemp_ST110_Avg_SourceID_1_QC_1.csv'),
    skiprows=56,
    index_col=0,
    )
gamut_hc2s3 = pd.read_csv(os.path.join(environment, 'gamut/tonygrove', f'LR_TG_C_RH_HC2S3_Avg_SourceID_1_QC_1.csv'), parse_dates=['date'], index_col='date')

In [ ]:
# Load gdriver credentials
credentials = get_credentials()

# Get the ST110 data
gamut_st110 = getCSVtoPandas('TonyGrove/LR_TG_C_AirTemp_ST110_Avg_SourceID_1_QC_1.csv',credentials,skiprows=56)

# Change index to DateTimeUTC
gamut_st110['DateTime'] = pd.to_datetime(gamut_st110['LocalDateTime'])
gamut_st110 = gamut_st110.set_index('DateTime')

# Convert -9999 to NaN
gamut_st110 = gamut_st110.replace(-9999, np.nan)

In [ ]:
# Extract a subset of the data for the time period of interest
start_date = '2015-01-01 00:00:00'
end_date = '2015-03-01 00:00:00'
gamut_st110_month = gamut_st110.loc[start_date:end_date]

# Localize datetime to america/denver timezone
gamut_st110_month.index = gamut_st110_month.index.tz_localize('America/Denver')

## Comparisons

In [ ]:
# Plot Tair and compare to ST110 data
fig, ax = plt.subplots(figsize=(12, 6))
forcing_Tair['Tair_C'].plot(ax=ax, label='Forcing Tair (C)', color='r', linestyle='-')
gamut_st110_month['AirTemp_ST110_Avg'].plot(ax=ax, label='GAMUT Tair (C)', color='b', linestyle='-')
nldas_Tair['Tair_C'].loc[start_date:end_date].plot(ax=ax, label='NLDAS Tair (C)', color='k', linestyle='--')
snotel.data.loc[start_date:end_date,'T_max_C'].plot(ax=ax, label='SNOTEL Tair (C)', color='r', marker='*', linestyle='None')
snotel.data.loc[start_date:end_date,'T_min_C'].plot(ax=ax, label='SNOTEL Tair (C)', color='b', marker='*', linestyle='None')
ax.legend()
# fig, ax = plt.subplots(figsize=(12, 6))
# forcing_Tair['Tair_C'].plot(ax=ax, label='Forcing Tair (C)', style='r')
# gamut_st110_month['AirTemp_ST110_Avg'].plot(ax=ax, label='GAMUT Tair (C)', style='b-')
# nldas_Tair['Tair_C'].loc[start_date:end_date].plot(ax=ax, label='NLDAS Tair (C)', style='k--')
# snotel.data.loc[start_date:end_date,'T_max_C'].plot(ax=ax, label='SNOTEL Tair (C)', style='r*')
# snotel.data.loc[start_date:end_date,'T_min_C'].plot(ax=ax, label='SNOTEL Tair (C)', style='b*')
# ax.legend()

In [ ]:
# Load Tair V0 for comparison
sitename = site.replace(' ', '').lower()
datapath_v0 = os.path.join(environment,'forcing',f'{sitename}_{start_date}_{end_date}_v0.1.0.nc')
forcing_Tair_v0 = get_forcing_values(datapath_v0, 'Tair', start_date, end_date)
forcing_Tair_v0['Tair_C'] = forcing_Tair_v0['Tair'] - 273.15
forcing_Tair_v0.index = forcing_Tair_v0.index.tz_localize('UTC').tz_convert('America/Denver')

In [ ]:
# Compare Tair V0 to V1
fig, ax = plt.subplots(figsize=(12, 6))
forcing_Tair['Tair_C'].plot(ax=ax, label='Forcing Tair V1 (C)', color='r', linestyle='-')
forcing_Tair_v0['Tair_C'].plot(ax=ax, label='Forcing Tair V0 (C)', color='r', linestyle='--')
gamut_st110_month['AirTemp_ST110_Avg'].plot(ax=ax, label='GAMUT Tair (C)', color='b', linestyle='-')


#